#### Scrape SEC filings

In [ ]:
url="https://efts.sec.gov/LATEST/search-index?category=form-cat2&ciks=0001535527&entityName=CrowdStrike%20Holdings%2C%20Inc.%20(CRWD)%20(CIK%200001535527)&forms=3%2C4%2C5&startdt=2021-10-28&enddt=2026-10-28"

In [ ]:
import requests
import pandas as pd
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Encoding": "gzip, deflate, br",
    "Origin": "https://www.sec.gov",
    "Referer": "https://www.sec.gov/",
}

# https://efts.sec.gov/LATEST/search-index?category=form-cat2&ciks=0001535527&entityName=CrowdStrike%20Holdings%2C%20Inc.%20(CRWD)%20(CIK%200001535527)&forms=3%2C4%2C5&startdt=2021-10-28&enddt=2026-10-28

params = {
    # "q": "glp", # search term
    "ciks":"0001535527",
    "dateRange": "custom",
    "category": "form-cat0",
    # "startdt": "2026-05-29",
    # "enddt": "2026-06-01",
    "forms": "4",
    "size": 100,  # max per page
    "from": 0,
}

all_hits = []

## loop through pages until we have all results
while True:
    r = requests.get("https://efts.sec.gov/LATEST/search-index", params=params, headers=headers)
    data = r.json()
    
    hits = data["hits"]["hits"]
    total = data["hits"]["total"]["value"]
    all_hits.extend(hits)
    
    print(f"Fetched {len(all_hits)} / {total}")
    
    if len(all_hits) >= total:
        break
    
    params["from"] += len(hits)
    time.sleep(0.2)  # be polite


## create a data frame 
rows = []
for h in all_hits:
    s = h["_source"]
    _id = h["_id"]
    id_part1 = _id.split(":")[0].replace("-", "")
    id_part2 = _id.split(":")[1]
    cik = int(s["ciks"][0])
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{id_part1}/{id_part2}"
    
    rows.append({
        "cik": cik,
        "display_name": s.get("display_names", [None]),
        "form_type": s.get("form"),
        "file_date": s.get("file_date"),
        "adsh": s.get("adsh"),
        "period_ending": s.get("period_ending"),
        "url": url,
    })

df = pd.DataFrame(rows)
df["file_date"] = pd.to_datetime(df["file_date"])
df = df.sort_values("file_date", ascending=False).reset_index(drop=True)
df.shape

In [ ]:
df.head()

In [ ]:
df['url'][0]